## Synthetic Data Corruption

To test the robustness of the preprocessing pipeline, a copy of the original
dataset is deliberately corrupted using controlled noise injection.

The original Google Patents extract remains unchanged.

Injected issues:
- Missing values
- Duplicate rows
- Whitespace inconsistencies
- Case inconsistencies
- Invalid dates
- Malformed categorical labels
- Extreme numeric outliers

A fixed random seed is used so that the experiment is reproducible.

In [96]:
import numpy as np
import pandas as pd
import json

# ============================================================
# CREATE A COPY
# ============================================================

dirty_df = raw_df.copy(deep=True)

# Reproducible random generator
rng = np.random.default_rng(42)

print("Original shape:", raw_df.shape)
print("Dirty copy shape:", dirty_df.shape)

Original shape: (80566, 20)
Dirty copy shape: (80566, 20)


In [98]:
# ============================================================
# 1. INJECT MUCH MORE VARIED MISSING VALUES
# ============================================================

missing_rates = {
    "application_number_formatted": 0.0002,   # 0.02%
    "filing_date":                  0.0008,   # 0.08%
    "priority_date":                0.0025,   # 0.25%
    "title_raw":                    0.0055,   # 0.55%
    "abstract_raw":                 0.0095,   # 0.95%
    "inventors_raw":                0.0135,   # 1.35%
    "assignees_raw":                0.0240    # 2.40%
}

for col, rate in missing_rates.items():

    n = int(len(dirty_df) * rate)

    indices = rng.choice(
        dirty_df.index,
        size=n,
        replace=False
    )

    dirty_df.loc[indices, col] = np.nan

    print(
        f"{col}: injected {n:,} missing values "
        f"({rate * 100:.2f}%)"
    )

application_number_formatted: injected 16 missing values (0.02%)
filing_date: injected 64 missing values (0.08%)
priority_date: injected 201 missing values (0.25%)
title_raw: injected 443 missing values (0.55%)
abstract_raw: injected 765 missing values (0.95%)
inventors_raw: injected 1,087 missing values (1.35%)
assignees_raw: injected 1,933 missing values (2.40%)


In [99]:
# ============================================================
# 2. ADD DUPLICATE ROWS
# ============================================================

n_duplicates = 250

duplicate_rows = dirty_df.sample(
    n=n_duplicates,
    random_state=42
)

dirty_df = pd.concat(
    [dirty_df, duplicate_rows],
    ignore_index=True
)

print("Duplicate rows added:", n_duplicates)
print("New shape:", dirty_df.shape)

Duplicate rows added: 250
New shape: (80816, 20)


In [100]:
# ============================================================
# 3. WHITESPACE INCONSISTENCIES
# ============================================================

n_whitespace = 400

indices = rng.choice(
    dirty_df.index,
    size=n_whitespace,
    replace=False
)

dirty_df.loc[indices, "country_code"] = (
    "  "
    + dirty_df.loc[indices, "country_code"].astype(str)
    + " "
)

print("Whitespace inconsistencies injected.")

Whitespace inconsistencies injected.


In [101]:
# ============================================================
# 4. CASE INCONSISTENCIES
# ============================================================

n_case = 400

indices = rng.choice(
    dirty_df.index,
    size=n_case,
    replace=False
)

dirty_df.loc[indices, "kind_code"] = (
    dirty_df.loc[indices, "kind_code"]
    .astype(str)
    .str.lower()
)

print("Case inconsistencies injected.")

Case inconsistencies injected.


In [102]:
# ============================================================
# 5. MALFORMED CATEGORICAL LABELS
# ============================================================

category_errors = [
    "USA",
    "U.S.",
    "United States",
    "us",
    "US ",
    "UNKNOWN"
]

n_category_errors = 300

indices = rng.choice(
    dirty_df.index,
    size=n_category_errors,
    replace=False
)

dirty_df.loc[indices, "country_code"] = rng.choice(
    category_errors,
    size=n_category_errors
)

print("Malformed category labels injected.")

Malformed category labels injected.


In [103]:
# ============================================================
# 6. INVALID DATE VALUES
# ============================================================

invalid_dates = [
    0,
    99999999,
    20221340,   # impossible month/day
    20220230,   # impossible date
    19000101,   # unrealistic for our study
    20501231    # future date
]

n_invalid_dates = 250

for col in ["filing_date", "priority_date"]:

    indices = rng.choice(
        dirty_df.index,
        size=n_invalid_dates,
        replace=False
    )

    dirty_df.loc[indices, col] = rng.choice(
        invalid_dates,
        size=n_invalid_dates
    )

print("Invalid dates injected.")

Invalid dates injected.


In [104]:
# ============================================================
# 7. CREATE NUMERIC VARIABLES FOR OUTLIER TESTING
# ============================================================

dirty_df["claims_raw_length"] = (
    dirty_df["claims_raw"]
    .fillna("")
    .astype(str)
    .str.len()
)


def citation_count(value):
    if pd.isna(value):
        return np.nan

    try:
        data = json.loads(value)

        if isinstance(data, list):
            return len(data)

        return np.nan

    except Exception:
        return np.nan


dirty_df["citation_count_raw"] = (
    dirty_df["citations_raw"]
    .apply(citation_count)
)

dirty_df[
    ["claims_raw_length", "citation_count_raw"]
].describe()

,claims_raw_length,citation_count_raw
count,8.081600e+04,80816.000000
mean,1.025303e+04,52.228569
std,1.274866e+04,159.854898
min,6.100000e+02,0.000000
25%,7.139000e+03,11.000000
50%,9.197000e+03,20.000000
75%,1.185600e+04,38.250000
max,2.698347e+06,9253.000000


In [105]:
# ============================================================
# 8. EXTREME NUMERIC OUTLIERS
# ============================================================

# Claims length outliers
n_claim_outliers = 100

indices = rng.choice(
    dirty_df.index,
    size=n_claim_outliers,
    replace=False
)

dirty_df.loc[
    indices,
    "claims_raw_length"
] *= rng.integers(
    20,
    100,
    size=n_claim_outliers
)


# Citation outliers
n_citation_outliers = 100

indices = rng.choice(
    dirty_df.index,
    size=n_citation_outliers,
    replace=False
)

dirty_df.loc[
    indices,
    "citation_count_raw"
] *= rng.integers(
    20,
    80,
    size=n_citation_outliers
)


print("Synthetic outliers injected.")

Synthetic outliers injected.


In [106]:
# ============================================================
# 9. MALFORMED JSON — SMALL NUMBER ONLY
# ============================================================

n_bad_json = 100

indices = rng.choice(
    dirty_df.index,
    size=n_bad_json,
    replace=False
)

dirty_df.loc[
    indices,
    "assignees_raw"
] = '{"broken_json": '

print("Malformed JSON rows injected:", n_bad_json)

Malformed JSON rows injected: 100


In [107]:
# ============================================================
# 10. SYNTHETIC DATA QUALITY CHECK
# ============================================================

print("=" * 50)
print("SYNTHETIC DIRTY DATASET")
print("=" * 50)

print("\nRows:")
print(f"Original: {len(raw_df):,}")
print(f"Dirty:    {len(dirty_df):,}")

print("\nDuplicate publication numbers:")
print(
    dirty_df["publication_number"]
    .duplicated()
    .sum()
)

print("\nCountry-code categories:")
print(
    dirty_df["country_code"]
    .value_counts(dropna=False)
    .head(15)
)

print("\nKind codes:")
print(
    dirty_df["kind_code"]
    .value_counts(dropna=False)
)

print("\nMissing values:")
display(
    dirty_df
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

SYNTHETIC DIRTY DATASET

Rows:
Original: 80,566
Dirty:    80,816

Duplicate publication numbers:
250

Country-code categories:
country_code
US               80117
  US               399
United States       68
USA                 50
U.S.                49
US                  47
UNKNOWN             46
us                  40
Name: count, dtype: int64

Kind codes:
kind_code
B2    71922
B1     8494
b2      358
b1       42
Name: count, dtype: int64

Missing values:


pct_number                      73022
assignees_raw                    1937
inventors_raw                    1092
abstract_raw                      771
title_raw                         443
priority_date                     202
filing_date                        63
application_number_formatted       16
claims_raw_length                   0
raw_grant_year                      0
citations_raw                       0
cpc_raw                             0
claims_raw                          0
publication_number                  0
application_number                  0
dtype: int64

In [56]:
# ============================================================
# SAVE SYNTHETIC DIRTY DATASET
# ============================================================

dirty_path = (
    PROCESSED_DIR /
    "Combined_data.parquet"
)

dirty_df.to_parquet(
    dirty_path,
    index=False
)

print("Saved to:")
print(dirty_path)

Saved to:
/Users/janakdobariya/Bramha/NLP_Engineering/BA/ai_patent_business_analytics/Data/processed/01_synthetic_dirty_patents.parquet
